# 10｜價格動能效應：強者恆強的學術基礎

**學術來源：**
- Jegadeesh & Titman (1993), *Returns to Buying Winners and Selling Losers*, Journal of Finance
- Carhart (1997): 加入動能因子（MOM）到三因子模型
- Daniel & Moskowitz (2016): *Momentum Crashes*，解釋動能崩潰機制

## 台積電漲了三個月，你敢追嗎？

很多人的直覺是：漲多了就是快要跌了。「均值回歸嘛——什麼東西漲太多最後都會跌回來。」

這個直覺沒有完全錯。但時間尺度很重要。

**短期（3–12 個月）的規律恰恰相反。**

1993 年，Jegadeesh 和 Titman 發現：
過去 3–12 個月報酬排名最高的股票，接下來一個月繼續領先的機率統計上顯著高於平均。
強者恆強——動能效應，在數據上是真實的。

所以「均值回歸」和「動能效應」哪個對？**兩個都對，只是時間尺度不同：**
- 動能：3–12 個月
- 均值回歸：3–5 年

問題是，動能策略有一個危險特性叫「**動能崩潰（Momentum Crash）**」——
在市場急速反轉時，動能策略會在極短時間內損失慘重。

這一章，我們把動能效應的全貌攤開來看，包括報酬、風險、以及個人投資者能不能實際用。

## 🎯 學習目標
1. 理解 Jegadeesh & Titman (1993) 的價格動能效應及其經濟解釋
2. 分析 WML（Winner Minus Loser）因子的歷史績效與動能崩潰
3. 學會評估動能策略的實務可行性（交易成本與崩潰風險）

## 1｜動能效應：理論與直覺

**核心發現（Jegadeesh & Titman 1993）：**
> 過去 3–12 個月的強勢股（Winners），在未來 3–12 個月繼續跑贏弱勢股（Losers）。

**WML 因子 = Winners − Losers**
- Winners：過去12個月（去掉最近1個月）報酬排名前30%的股票
- Losers：排名後30%的股票

**為什麼動能存在？（學術爭議）**
| 解釋 | 機制 |
|------|------|
| 行為金融（主流）| 投資人對好消息反應不足，慢慢追漲 |
| 資訊擴散 | 消息從分析師→機構→散戶逐步傳遞 |
| 趨勢追隨 | 動能吸引更多動能追隨者，自我實現 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = [
    'Microsoft YaHei', 'SimHei', 'Heiti TC',
    'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif'
]
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from scipy import stats
import warnings, pathlib
warnings.filterwarnings('ignore')

DATA_DIR = pathlib.Path('data')
DATA_DIR.mkdir(exist_ok=True)

MOM_CSV = DATA_DIR / 'ff_momentum.csv'
FF3_CSV = DATA_DIR / 'ff3_factors.csv'

# 載入動能因子
if MOM_CSV.exists():
    mom_df = pd.read_csv(MOM_CSV, index_col=0, parse_dates=True)
    print(f'動能因子從快取載入：{len(mom_df)} 筆月度資料')
else:
    import pandas_datareader.data as web
    raw = web.DataReader('F-F_Momentum_Factor', 'famafrench', start='1927-01')[0]
    mom_df = raw / 100
    mom_df.index = pd.to_datetime(mom_df.index.to_timestamp())
    mom_df.to_csv(MOM_CSV)
    print(f'動能因子下載完成：{len(mom_df)} 筆月度資料')

# 載入 FF3（用來對比）
if FF3_CSV.exists():
    ff3 = pd.read_csv(FF3_CSV, index_col=0, parse_dates=True)
else:
    import pandas_datareader.data as web
    raw3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1927-01')[0]
    ff3 = raw3 / 100
    ff3.index = pd.to_datetime(ff3.index.to_timestamp())
    ff3.to_csv(FF3_CSV)

mom = mom_df.iloc[:, 0]  # WML 欄位
common = mom.index.intersection(ff3.index)
mom = mom.loc[common]
mkt = ff3.loc[common, 'Mkt-RF']

print(f'\n動能因子統計（{common[0].year}–{common[-1].year}）：')
ann_ret = mom.mean() * 12
ann_vol = mom.std() * np.sqrt(12)
sharpe  = ann_ret / ann_vol
print(f'年化報酬：{ann_ret*100:.2f}%')
print(f'年化波動：{ann_vol*100:.2f}%')
print(f'夏普比率：{sharpe:.3f}')

## 2｜動能 vs 其他因子：累積報酬比較

In [ ]:
# 對齊所有因子
factors_compare = {
    '動能 WML': mom,
    '市場 Mkt-RF': mkt,
    '價值 HML': ff3.loc[common, 'HML'],
    '規模 SMB': ff3.loc[common, 'SMB'],
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'動能 WML': '#E91E63', '市場 Mkt-RF': '#9E9E9E',
          '價值 HML': '#FF9800', '規模 SMB': '#9C27B0'}
lws    = {'動能 WML': 2.5, '市場 Mkt-RF': 1.2, '價值 HML': 1.5, '規模 SMB': 1.5}

# 累積報酬
for name, s in factors_compare.items():
    cum = (1 + s).cumprod()
    axes[0].plot(cum.index, cum, label=name,
                 color=colors[name], linewidth=lws[name],
                 alpha=0.9 if name == '動能 WML' else 0.65)

axes[0].set_yscale('log')
axes[0].set_title('各因子累積報酬比較（對數尺度）', fontsize=13, fontweight='bold')
axes[0].set_ylabel('累積倍數')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# 夏普比率比較
sharpes = {n: (s.mean()*12) / (s.std()*np.sqrt(12)) for n, s in factors_compare.items()}
names_sorted = sorted(sharpes, key=sharpes.get)
vals = [sharpes[n] for n in names_sorted]
bar_colors = [colors[n] for n in names_sorted]
bars = axes[1].barh(names_sorted, vals, color=bar_colors, alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
for bar, v in zip(bars, vals):
    axes[1].text(v + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{v:.2f}', va='center', fontsize=11)
axes[1].set_title('年化夏普比率比較', fontsize=13, fontweight='bold')
axes[1].set_xlabel('夏普比率')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('data/momentum_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3｜動能崩潰（Momentum Crash）

動能策略最大的風險：**在熊市結束後的急速反轉中，策略會大幅虧損。**

**為什麼？**
- 動能策略在熊市末期持有「空方（Losers）」——這些往往是高 β 的爛股票
- 當市場急速反轉（如2009年3月），高 β 股暴漲，空方部位大幅虧損
- Daniel & Moskowitz (2016)：動能崩潰是可預測的，發生在隱含波動率高且市場剛從低點反彈後

In [ ]:
# 計算滾動12個月動能報酬 + 識別崩潰期
mom_annual = mom.rolling(12).apply(lambda x: (1+x).prod() - 1)

# 找出最差的12個月期間（rolling 12M < -20%）
crash_mask = mom_annual < -0.20

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# 上圖：動能因子累積報酬 + 崩潰期標注
cum_mom = (1 + mom).cumprod()
axes[0].plot(cum_mom.index, cum_mom, color='#E91E63', linewidth=1.5)
axes[0].set_yscale('log')

# 標注崩潰期（月報酬 < -10%）
for date, val in mom[mom < -0.10].items():
    axes[0].axvline(date, color='red', alpha=0.3, linewidth=0.8)

axes[0].set_title('WML 動能因子累積報酬（紅線 = 單月跌超10%）', fontsize=12, fontweight='bold')
axes[0].set_ylabel('累積倍數（對數）')
axes[0].grid(alpha=0.3)

# 下圖：滾動12個月報酬
axes[1].bar(mom_annual.index, mom_annual * 100,
            color=['#F44336' if v < 0 else '#4CAF50' for v in mom_annual],
            alpha=0.7, width=20)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axhline(-20, color='red', linewidth=1, linestyle='--', label='-20%崩潰門檻')
axes[1].set_title('滾動12個月動能報酬', fontsize=12, fontweight='bold')
axes[1].set_ylabel('12個月報酬 (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('data/momentum_crash.png', dpi=150, bbox_inches='tight')
plt.show()

# 最差月份
print('動能因子最差單月 Top 10：')
print(mom.nsmallest(10).map(lambda x: f'{x*100:.1f}%').to_string())

## 4｜這跟你有什麼關係？

### 動能策略的實務考量

**① 動能有效，但散戶難以執行**

WML 因子需要每月或每季換手，交易成本對報酬影響顯著：
- Novy-Marx & Velikov (2016)：考慮買賣價差後，許多動能策略報酬大幅縮水
- 機構投資者有規模優勢；散戶建議透過 ETF 捕捉動能溢酬

**② 動能崩潰不可忽視**

動能是少數「有系統性崩潰風險」的因子：
- 2009年3–5月：WML 單月 -20%、-22%
- 沒有其他因子有如此集中的尾部風險
- 如果你無法承受「指數大漲但動能策略大虧」，動能策略不適合你

**③ 實務工具**

| 工具 | 說明 |
|------|------|
| **MTUM**（iShares MSCI USA Momentum Factor） | 大型股動能 ETF |
| **QMOM**（Alpha Architect Quantitative Momentum） | 純動能策略，換手率較高 |
| **個股方法** | 用 52週高點附近、相對強度 > 80 作為初步篩選 |

**④ 組合搭配：動能 + 價值的矛盾**

動能（追強）和價值（買低）天生相反，兩者負相關。
搭配使用可以降低整體策略波動：
```
配置範例：50% 價值（HML/RMW）+ 50% 動能（WML）
```

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| 動能效應 | 過去3–12個月強勢股，未來3–12個月繼續跑贏 |
| WML 夏普比率 | 歷史上是所有因子中最高之一 |
| 動能崩潰 | 熊市大反轉時（如2009年），策略可能單月虧損20%+ |
| 交易成本 | 高換手率使散戶難以實際獲利（機構更有優勢）|
| 個人應用 | 使用動能 ETF（MTUM、QMOM）代替自行操作 |

> 🎓 **課程完結**：你已走過從市場估值到因子投資的完整實證金融旅程！
> 每一章的「本章重點摘要」就是你的帶走清單。